# 02 — Lists, `for` loops, and `if` / `else`

**Goal:** be able to take a list of trade P&Ls and answer questions about it — how many winners, total profit, biggest loser, etc. — using only Python and a `for` loop. This is genuinely 80% of what `engine/metrics.py` does at its core, just dressed up in numpy.

**Same rules as last time:**
- Run every cell. Predict the output before you run it.
- Type the exercises by hand. No copy-paste.
- If you're stuck, try, get an error, read the error, ask me.

---
## 1. Lists — sequences of values

**Beginner terms:** a list is a row of boxes, in order. Each box can hold any value (int, float, string, bool, even another list). You write a list with square brackets `[ ]` and commas between items.

**Why quant finance uses lists:** every backtest produces a sequence — a list of trades, a list of P&Ls, a list of daily returns. Anything that comes in time order or trade order naturally lives in a list. Without lists you'd need a separate variable name for every trade (`trade_1`, `trade_2`, ... ) which obviously doesn't scale to 5000 trades.

**The math/finance concept:** in stats and finance, the fundamental object is a *sequence of observations*. Mean, standard deviation, max drawdown — they're all functions OF a sequence. The Python list is how we represent that sequence.

In [ ]:
# A list of P&Ls from 8 trades. Some winners, some losers.
pnls = [340, -180, 720, -95, -420, 1100, 250, -310]

print(pnls)
print("How many trades?", len(pnls))   # len() = length
print("Type:", type(pnls))

**Two new things just appeared:**
1. `[340, -180, 720, ...]` — square brackets create a list. Items separated by commas.
2. `len(pnls)` — built-in function that tells you how many items are in a list. You'll use `len()` constantly.

---
## 2. Indexing — getting one item out

**Beginner terms:** to get a single item from a list, use square brackets with the position number: `pnls[0]`. The first item is position 0, not 1. (This trips everyone up at first. Get used to it.)

**Why quant uses it:** to access "the most recent bar's close," "the trade before this one," "yesterday's stop level." Backtesting is full of "give me the value at position N" questions.

**Negative indexing:** `pnls[-1]` is the LAST item. `pnls[-2]` is the second to last. This is a Python convenience — you don't have to compute `len(pnls) - 1` to get the end.

In [ ]:
pnls = [340, -180, 720, -95, -420, 1100, 250, -310]

print("First trade (index 0):", pnls[0])
print("Second trade (index 1):", pnls[1])
print("Last trade (index -1):", pnls[-1])
print("Second to last (index -2):", pnls[-2])

In [ ]:
# Predict what this will print BEFORE running:
print(pnls[3])
print(pnls[-3])
print(pnls[len(pnls) - 1])    # equivalent to pnls[-1]

**Slicing — getting a chunk:**

`pnls[start:end]` returns a *new* list with items from position `start` up to (but NOT including) `end`. The half-open interval is the source of half of all off-by-one bugs in Python.

In [ ]:
print("First 3 trades:", pnls[0:3])
print("Trades 2 through 4 (positions 2,3,4):", pnls[2:5])
print("Last 3 trades:", pnls[-3:])
print("All except last:", pnls[:-1])

---
## 3. Modifying lists

**Beginner terms:** unlike numbers, lists are *mutable* — you can change them in place. Add items, remove items, replace items.

**Why finance uses it:** the backtester maintains a running list of open trades and closed trades. As trades are entered and exited, the lists grow. `.append()` is what does that growth.

In [ ]:
trades = []                # an empty list
print("Start:", trades, "len:", len(trades))

trades.append(340)         # add 340 to the end
trades.append(-180)
trades.append(720)
print("After 3 trades:", trades, "len:", len(trades))

In [ ]:
# Replacing an item by index:
pnls = [340, -180, 720, -95]
pnls[1] = 0    # the second trade was actually breakeven
print(pnls)

---
## 4. The `for` loop — "do this to every item"

**Beginner terms:** a `for` loop walks through a list, one item at a time, and runs whatever code you put inside the loop on each item. The variable name between `for` and `in` is just a label for "the current item this time around the loop."

**Why quant uses it:** every backtest is fundamentally "for each bar, decide what to do." Every metric is "for each trade, accumulate something." Loops are the engine of the engine.

**The syntax is fussy:**
- The line ends with a colon `:`
- The body of the loop is **indented** (4 spaces). Indentation is how Python knows what's inside the loop and what's outside. **This matters in Python — it's not just style, it's syntax.**

In [ ]:
pnls = [340, -180, 720, -95, -420, 1100, 250, -310]

for pnl in pnls:
    print("This trade made:", pnl)

print("--- Loop done.")

**What just happened, mechanically:**
1. Python looked at `pnls` and grabbed the first item (340).
2. It put 340 into a variable called `pnl` (the name we picked between `for` and `in`).
3. It ran the indented body — `print("This trade made:", pnl)`.
4. Then it grabbed the next item (-180), put it in `pnl`, ran the body again. And so on.
5. After the last item, the loop ended and Python ran the next non-indented line.

The variable name `pnl` is your choice. We could have written `for x in pnls:` or `for trade in pnls:` — Python doesn't care. Pick a name that reads well.

**The accumulator pattern — the most useful loop trick:**

Start with a variable holding 0 (or some starting value). Each time around the loop, update it. After the loop, you have your answer.

In [ ]:
pnls = [340, -180, 720, -95, -420, 1100, 250, -310]

total = 0
for pnl in pnls:
    total += pnl   # remember: total = total + pnl

print("Total P&L:", total)

**Why this matters:** this is *literally* how you compute net P&L from a list of trades. And the same pattern (start at 0, walk the list, accumulate) shows up everywhere — running sum, running max, running drawdown, equity curve construction. You will write this pattern hundreds of times.

Python actually has a built-in shortcut: `sum(pnls)` does the same thing in one call. But understanding the manual loop is what lets you do *anything* per-trade, not just sum.

In [ ]:
print("Sum the easy way:", sum(pnls))

---
## 5. `if` / `else` — branching on a condition

**Beginner terms:** `if` runs a block of code only when a condition is True. `else` runs when the condition is False. `elif` ("else if") chains more conditions.

**Why finance uses it:** every entry rule, exit rule, halt check is an if. "If the stop got hit, close the position." "If we've lost 5 in a row, stop trading for the day." "If MFE > 2 ATR, take profit."

Same syntax rules as `for`: line ends with colon, body is indented.

In [ ]:
pnl = 340

if pnl > 0:
    print("Winner!")
else:
    print("Loser.")

In [ ]:
# Three-way branch with elif:
pnl = 0

if pnl > 0:
    print("Winner")
elif pnl < 0:
    print("Loser")
else:
    print("Breakeven")

**Important:** `if` is followed by something that evaluates to True or False — that's a *bool* (remember notebook 01). `pnl > 0` returns a bool. `is_in_trade` (already a bool) works directly. So does `len(trades) > 0`.

---
## 6. Combining `for` + `if` — the bread and butter

**Beginner terms:** a `for` with an `if` inside lets you walk a list and only do something when a condition is met. Skip everything else.

**Why this matters:** "how many winning trades?", "sum just the losers," "count days the strategy was halted." These are all *for-with-an-if-inside* patterns.

In [ ]:
pnls = [340, -180, 720, -95, -420, 1100, 250, -310]

n_winners = 0
n_losers  = 0

for pnl in pnls:
    if pnl > 0:
        n_winners += 1
    else:
        n_losers += 1

print("Winners:", n_winners)
print("Losers: ", n_losers)
print("Win rate:", n_winners / len(pnls) * 100, "%")

**Notice the indentation:**
- `for` body is indented 4 spaces
- `if` is *inside* the for, so its body is indented 8 spaces
- `n_winners += 1` is the body of the `if`, indented 8 spaces from the start of the line

Indentation level = nesting level. A line indented N spaces belongs to the block at indent N-4.

In [ ]:
# A more interesting one: gross profit and gross loss separately.
# (This is what 'profit factor' uses — gross_profit / gross_loss.)

pnls = [340, -180, 720, -95, -420, 1100, 250, -310]

gross_profit = 0
gross_loss   = 0

for pnl in pnls:
    if pnl > 0:
        gross_profit += pnl
    elif pnl < 0:
        gross_loss += pnl     # gross_loss accumulates negative numbers

print("Gross profit:", gross_profit)
print("Gross loss:  ", gross_loss)
print("Profit factor:", gross_profit / abs(gross_loss))

**Worth pausing on this one.** You just wrote — from primitives — the *profit factor* calculation. Profit factor is the ratio of total winners to total losers (in absolute value). PF > 1 means the strategy makes money; PF = 1 is breakeven; PF < 1 loses.

Look back at `engine/metrics.py` lines 187–189 once you're done — you'll recognise this exact computation, except it's vectorised in numpy:
```python
gross_profit = winners.sum() if len(winners) > 0 else 0.0
gross_loss = abs(losers.sum()) if len(losers) > 0 else 0.0
profit_factor = gross_profit / (gross_loss + _EPS)
```

Same idea, different machinery.

---
## 7. The running maximum (worth seeing once)

**Why:** drawdown is computed against a *running peak* — the highest equity value seen so far. That requires keeping a variable that updates each bar to be the max of itself and the latest equity. This is the second most common loop pattern in finance code (after sum).

In [ ]:
pnls = [340, -180, 720, -95, -420, 1100, 250, -310]
starting_balance = 100000

balance = starting_balance
running_peak = starting_balance
max_drawdown = 0

for pnl in pnls:
    balance += pnl
    if balance > running_peak:
        running_peak = balance
    drawdown = balance - running_peak    # always <= 0
    if drawdown < max_drawdown:
        max_drawdown = drawdown
    print(f"Balance {balance}, peak {running_peak}, dd {drawdown}")

print("\nWorst drawdown over the run: $", max_drawdown)

Read the loop carefully. Each bar:
1. Apply the new P&L to balance.
2. If new balance is higher than any previous peak, update the peak.
3. Drawdown = how far below the peak we are right now (always ≤ 0).
4. If this drawdown is worse than any drawdown we've seen, update max_drawdown.

**This is exactly the algorithm in `engine/metrics.py`** lines 86–88, except numpy does it in two vectorised lines instead of a loop. Same logic underneath.

---
## Exercises

Type by hand. Predict before running.

### Exercise 1 — count and sum

Given `pnls = [120, -50, -75, 200, 90, -30, -110, 450, -20, 60]`:
- Use a `for` loop and an `if` to count how many losing trades there are.
- Use the same loop (or a separate one) to compute the sum of just the losing trades.
- Print both.

In [ ]:
# your code here


### Exercise 2 — average win

Same `pnls` list. Compute the **average winning trade** (sum of winners divided by number of winners). Print it.

Hint: you'll need two accumulators — one for sum, one for count — both updated only when `pnl > 0`.

In [ ]:
# your code here


### Exercise 3 — biggest single loser

Find the worst (most negative) trade in the list. Print it.

Hint: same pattern as the running maximum, but tracking a running *minimum*. Start your tracker variable at 0 (or at `pnls[0]`), and update it any time you see a smaller value.

In [ ]:
# your code here


### Exercise 4 — equity curve

Start with `balance = 100000` and the same `pnls` list. Build a list `equity_curve` that holds the running balance after each trade. So if pnls is `[120, -50, ...]` and starting balance is 100000, equity_curve should start `[100120, 100070, ...]`.

Hint: `equity_curve = []` then `.append(balance)` inside the loop after each update.

Print the final list and its length.

In [ ]:
# your code here


### Exercise 5 — consecutive losses

This one is harder. Walk the `pnls` list and find the **maximum number of consecutive losing trades**.

Strategy:
- Keep a counter `current_streak` that starts at 0.
- Keep `max_streak` that starts at 0.
- For each trade: if it's a loser, add 1 to `current_streak`; if it's a winner, reset `current_streak` to 0.
- After updating `current_streak`, if it's bigger than `max_streak`, update `max_streak`.

Print the answer.

(For the given list `[120, -50, -75, 200, 90, -30, -110, 450, -20, 60]`, the answer is 2 — there's a streak of `[-50, -75]` and a streak of `[-30, -110]`. Predict before running.)

In [ ]:
# your code here


---
## When you're done

Tell me you've finished and let me know:
1. Anywhere you got stuck or had to retry.
2. Anything that's still hazy.
3. If exercise 5 felt different in difficulty from 1–4.

Next notebook (`03_functions_and_dicts.ipynb`) will cover **functions** (reusable named blocks of code, like `_profit_factor()` in `metrics.py`) and **dictionaries** (key-value lookups, like the config dicts all over `engine/`).